# HR Attrition Risk Prediction: End-to-End Workflow

Run cells in order. Training requires the MLflow server configured by `MLFLOW_TRACKING_URI`.


In [1]:
from pathlib import Path
import os, sys, joblib, pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from app.config import settings
from app.data import (sync_data_from_s3, validate_source_data, preprocess_data, feature_engineering, feature_selection, exploratory_data_analysis)
from app.train import split_data, handle_imbalance_data, train_and_track_models
from app.evaluate import evaluate_classifier, evaluate_risk_bands
from app.features import FINAL_FEATURES, TARGET_COLUMN
from app.inference import get_inference_services
from app.explain import generate_llm_explanation
from app.schemas import AttritionInput, ExplainInput

print(f'Project root: {ROOT}')
print(f'Data source: {settings.DATA_SOURCE}')
print(f'MLflow URI: {settings.MLFLOW_TRACKING_URI}')

Project root: D:\Cybersoft__Course\MLOps_01\buoi10\Đề bài\04\hr_attrition_risk_prediction
Data source: s3
MLflow URI: http://localhost:5001


## 1. Ingest and validate source data

`sync_data_from_s3` skips safely for local data. `validate_source_data` performs the temporal joins and writes `data/merged/merged_data.parquet`.


In [2]:
ingestion_report = sync_data_from_s3()
validation_report = validate_source_data()
display(pd.DataFrame([ingestion_report, validation_report]))

merged_df = pd.read_parquet(settings.MERGED_DATA_PATH)
display(merged_df.head())
print(f'Merged dataset shape: {merged_df.shape}')

,status,bucket,prefix,files,data_dir,validated_at,rows,columns,source_file
0,downloaded,baitap-buoi9-bucket,data/,"[attrition_labels.csv, employees.csv, engageme...",data\raw,NaN,NaN,NaN,NaN
1,passed,NaN,NaN,NaN,NaN,2026-10-10T16:28:27.830633+00:00,128540.0,40.0,6.0


,employee_id,snapshot_date,department,job_level,label,role,location,employment_type,hire_date,manager_id,...,peer_feedback_score,review_sentiment_score,week_start,tasks_completed,late_tasks,overtime_hours,meeting_hours,focus_hours,after_hours_messages,absent_days
0,E000001,2025-01-31,Customer Success,3,0,Senior CSM,Ho Chi Minh City,Full-time,2016-01-06,M00435,...,4.27,0.876,2025-01-27,61,7,9.5,11.9,28.4,17,0
1,E000002,2025-01-31,Operations,3,0,Senior Operations Specialist,Ho Chi Minh City,Full-time,2023-11-13,M00204,...,3.11,0.633,2025-01-27,47,1,8.5,10.6,26.5,8,0
2,E000003,2025-01-31,Marketing,2,0,Marketing Specialist,Ho Chi Minh City,Contract,2017-02-09,M00367,...,3.46,0.965,2025-01-27,27,2,8.7,7.9,23.0,9,0
3,E000004,2025-01-31,Customer Success,1,0,CS Associate,Da Nang,Full-time,2022-12-19,M00795,...,3.75,0.852,2025-01-27,52,4,10.9,1.3,27.9,12,0
4,E000005,2025-01-31,Sales,1,0,Sales Associate,Ha Noi,Full-time,2016-02-29,M00460,...,3.04,0.701,2025-01-27,51,5,10.4,4.7,31.0,4,0


Merged dataset shape: (128540, 40)


## 2. Explore, preprocess, and select features

EDA files are written to `artifacts/eda_charts`; feature selection writes the model contract to `data/processed/feature_selected.parquet`.


In [ ]:
eda_report = exploratory_data_analysis()
preprocessing_report = preprocess_data()
engineering_report = feature_engineering()
selection_report = feature_selection()
display(pd.DataFrame([eda_report, preprocessing_report, engineering_report, selection_report]))

feature_df = pd.read_parquet(settings.FEATURE_SELECTED_PATH)
display(feature_df.head())
print(f'Selected feature dataset shape: {feature_df.shape}')

## 3. Split, balance, and train models

`train_and_track_models` logs nested MLflow runs, selects the highest-ROC-AUC candidate, and writes the model and serving metadata.


In [ ]:
split_report = split_data()
balance_report = handle_imbalance_data()
display(pd.DataFrame([split_report, balance_report]))

train_df = pd.read_parquet(settings.TRAIN_BALANCED_PATH)
test_df = pd.read_parquet(settings.TEST_BALANCED_PATH)
training_summary = train_and_track_models(train_df, test_df)
metrics_df = pd.DataFrame(training_summary['full_metrics']).sort_values('roc_auc', ascending=False)
display(metrics_df)
print(f"Best model: {training_summary['best_model']['Model']}")

## 4. Evaluate the persisted best model


In [ ]:
pipeline = joblib.load(settings.BEST_MODEL_PATH)
X_test, y_test = test_df[FINAL_FEATURES], test_df[TARGET_COLUMN]
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)
metrics = evaluate_classifier(y_test, y_pred, y_proba)
risk_band_report = evaluate_risk_bands(y_test, y_proba)
display(pd.DataFrame([metrics]))
display(pd.DataFrame(risk_band_report['bands_summary']).T)

## 5. Score an employee and generate a guardrailed explanation

The explanation service uses the configured LLM when available, otherwise its deterministic safe fallback.


In [ ]:
sample_row = test_df.dropna(subset=FINAL_FEATURES).iloc[0]
sample_payload = sample_row[FINAL_FEATURES].to_dict()
attrition_input = AttritionInput(**sample_payload)
prediction = get_inference_services().predict_one(attrition_input)
display(pd.DataFrame([prediction.model_dump(exclude={'top_drivers'})]))
display(pd.DataFrame([driver.model_dump() for driver in prediction.top_drivers]))

explain_input = ExplainInput(
    churn_probability=prediction.churn_probability,
    risk_band=prediction.risk_band,
    top_drivers=[driver.model_dump() for driver in prediction.top_drivers],
    department=sample_payload['department'],
    role=sample_payload['role'],
)
explanation = generate_llm_explanation(explain_input)
display(explanation)